# Monitoramento de germinação e crescimento de mudas

Notebook alinhado ao relatório do projeto: **detecção de mudas** (bandejas/vasos), **contagem estimada de folhas/galhos** por recorte, e base para **evolução temporal**.

**Fluxo recomendado no relatório**
- Roboflow para anotação/versionamento/export (YOLO).
- Treino principal em **Google Colab** (GPU).
- Modelo 1: detecção (**RF-DETR Small** no relatório; aqui usamos **YOLO11** como baseline direto no Ultralytics — opcional RF-DETR mais abaixo).
- Modelo 2: regressão/classificação de **nº de folhas** em cada recorte de muda.

**Antes de rodar**: em Colab use *Runtime → Change runtime type → GPU* (T4 ou superior ajuda).

## 1. Ambiente e GPU
Verifique se o runtime está usando CUDA.

In [ ]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Instalação de dependências

**Importante (Colab + GPU):** instalar só `ultralytics` costuma puxar `torch` **+cpu** e o log fica `device=cpu` para sempre. O bloco de código instala **PyTorch com CUDA** antes e, se for preciso, reforça o build no fim.

1. No Colab: **Ambiente → Alterar tipo de ambiente → acelerador GPU (T4, etc.)** *antes* de instalar.
2. Depois de instalar, se a célula de **verificação** ainda disser `+cpu`, usa **Ambiente → Reiniciar ambiente** e volta a correr a verificação (e o resto) **sem** reinstalar tudo, a menos que apagues o ambiente.

Componentes: **ultralytics** (YOLO11), **roboflow** (opcional), **supervision**, **pandas** / **matplotlib**.

In [ ]:
# 1) PyTorch com CUDA (Colab costuma usar CUDA 12.x; cu124 cobre a maioria das sessões)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124
# 2) Resto (ultralytics pode tentar instalar torch+cpu por dependência)
!pip install -q ultralytics roboflow supervision matplotlib pandas pyyaml opencv-python-headless scikit-learn
# 3) Garantir que ficamos com torch CUDA por cima de qualquer dependência CPU
!pip install -q --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu124

### Verificação GPU (obrigatório após instalar)

Se `nvidia-smi` falhar, **não há GPU nesta sessão** (muda *Ambiente → Tipo de ambiente*). Se `torch` ainda mostrar `+cpu` após tudo, faz **Ambiente → Reiniciar ambiente** e corre **só** esta célula e a de treino (não precisas reinstalar a cada reinício, mas a primeira vez após `pip` o reinício assegura que o `import torch` carrega a build certa).

In [ ]:
import shutil
import subprocess

_smi = shutil.which("nvidia-smi")
if _smi is None:
    print("nvidia-smi não existe nesta sessão → VM sem drivers NVIDIA.")
    print("No Google Colab: Ambiente → Alterar tipo de ambiente → GPU (ex.: T4), depois Ambiente → Reiniciar ambiente.")
else:
    r = subprocess.run([_smi, "-L"], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        print(r.stdout)
    else:
        print("GPU não detetada via nvidia-smi. Reinicia com ambiente GPU ou verifica quota Colab.")

import torch
print("torch.__version__ =", torch.__version__)
print("torch.cuda.is_available() =", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    if "+cpu" in torch.__version__ or "cpu" in str(torch.__version__):
        print("AINDA com build CPU. Tenta: Ambiente → Reiniciar ambiente, depois corre de novo SÓ esta célula.")
    print("Se após reiniciar continuar em CPU, executa numa célula:\n"
          "!pip install -q --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu124\n"
          "e volta a reiniciar o ambiente.")

## 3. Preparar o dataset

Escolha **uma** das opções:

### A) ZIP exportado do Roboflow (YOLOv8/YOLO11)
Faça upload do `.zip` pelo menu à esquerda (ícone de pasta → **Upload**). Use o nome **`upload.zip`** para bater com o código (caminho típico: **`/content/upload.zip`**). Se usar outro nome, a célula ainda tenta usar o único `.zip` encontrado em `/content/`.

### B) Download direto via Roboflow Python
Preencha `ROBOFLOW_API_KEY`, `WORKSPACE`, `PROJECT`, `VERSION`. O código baixa e descompacta em `/content/data`.

**Nota sobre o `data.yaml`**: versões exportadas às vezes apontam `train: ../train/images`. O próximo bloco corrige caminhos relativos ao arquivo.

In [ ]:
from pathlib import Path
import zipfile

DATA_PARENT = Path("/content/data")
DATA_PARENT.mkdir(parents=True, exist_ok=True)


def encontrar_zip():
    candidatos = [
        Path("/content/upload.zip"),
        DATA_PARENT / "upload.zip",
        Path("/content/seedling.zip"),
    ]
    for p in candidatos:
        if p.exists():
            return p.resolve()
    outros = sorted(Path("/content").glob("*.zip"))
    return outros[0].resolve() if outros else None


def diagnosticar_upload(path: Path) -> None:
    """BadZipFile quase sempre = ficheiro que não é ZIP real (upload a meio, HTML, ou renomeado)."""
    sz = path.stat().st_size
    with open(path, "rb") as f:
        inicio = f.read(512)
    print(f"Tamanho em disco: {sz:,} bytes")
    print("Primeiros bytes (hex):", inicio[:16].hex())
    if sz == 0:
        raise RuntimeError("Ficheiro tem 0 bytes — o upload não terminou ou falhou.")
    if inicio.startswith(b"<!DOCTYPE") or inicio.startswith(b"<html"):
        raise RuntimeError(
            "O ficheiro parece HTML (página web), não um ZIP. "
            "Exporta de novo no Roboflow como YOLO e faz zip da pasta no teu PC."
        )
    if not inicio.startswith(b"PK"):
        raise RuntimeError(
            "Cabeçalho não é de ZIP (deveria começar por PK). "
            "Talvez seja .rar/.7z renomeado, ou corrupção no upload. "
            "No Mac: clica direito na pasta → Comprimir; envia o .zip gerado."
        )


ZIP_PATH = encontrar_zip()

if ZIP_PATH is None:
    print("Nenhum .zip em /content/")
    print("Alternativa fiável (Colab): na célula seguinte desta secção usa UPLOAD_INTERATIVO = True")
    print("Ou coloca o zip no Google Drive, monta o Drive e define ZIP_PATH ao caminho real.")
else:
    print("Usando:", ZIP_PATH)
    diagnosticar_upload(ZIP_PATH)
    try:
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            z.extractall(DATA_PARENT)
            lista = z.namelist()
            print("Primeiras entradas:", lista[:12])
            if len(lista) > 12:
                print("... total:", len(lista))
    except zipfile.BadZipFile as e:
        print("\n>>> Dicas: 1) Apaga upload.zip no painel e volta a enviar (espera a barra 100%).")
        print("   2) ZIPs grandes: usa Google Drive → copia para /content/data/ com !cp")
        print("   3) Ou executa a célula 'Upload interativo' abaixo com files.upload()")
        raise RuntimeError("ZIP inválido depois do diagnóstico — vê mensagens acima.") from e
    print("Extraído em:", DATA_PARENT)

yamls = sorted(DATA_PARENT.rglob("data.yaml"))
print("data.yaml encontrados:", [str(p) for p in yamls])
if not yamls:
    print("Aviso: nenhum data.yaml em", DATA_PARENT)
    print("Conteúdo:", [p.name for p in DATA_PARENT.iterdir()])

# Roboflow API (sem ZIP): descomenta e preenche
# from roboflow import Roboflow
# rf = Roboflow(api_key="SEU_API_KEY")
# rf.workspace("WORKSPACE").project("PROJECT").version(VERSAO).download("yolov11", location=str(DATA_PARENT / "roboflow_export"))

### Se o ZIP continuar inválido (`BadZipFile`)

O painel lateral do Colab às vezes **parte/corrompe** ficheiros grandes antes de ficarem 100% no disco.

**Tenta por ordem:**

1. **Célula seguinte** — upload com `files.upload()` (mais fiável que arrastar para a barra lateral).
2. **Google Drive**: carrega o zip para o Drive → no Colab *Montar o Drive* → `!cp "/content/drive/MyDrive/caminho/para/seedling.zip" /content/upload.zip` → volta à célula principal do ZIP.
3. **Roboflow**: descomenta o bloco na célula anterior e faz `download("yolov11", ...)` (zero zip manual).

Antes de subir de novo: no teu Mac, **abre o `.zip`** (duplo clique). Se o Preview não mostrar pastas/ficheiros, o zip está mal feito — volta a comprimir a pasta exportada pelo Roboflow.

In [ ]:
# --- Colab: upload fiável (executa só se upload.zip pela barra lateral falhou) ---
from pathlib import Path

try:
    from google.colab import files
except ImportError:
    print("Ignora esta célula fora do Google Colab.")
else:
    print("Escolhe o teu dataset.zip (export YOLO do Roboflow)...")
    uploaded = files.upload()
    if not uploaded:
        print("Nenhum ficheiro enviado.")
    for name, data in uploaded.items():
        dest = Path("/content/upload.zip")
        dest.write_bytes(data)
        print(f"Gravado: {dest} ({len(data):,} bytes)")
        if len(data) >= 4 and data[:2] != b"PK":
            print("AVISO: estes bytes não parecem ZIP (deveriam começar por PK). Verifica o ficheiro.")
    print("Agora volta a correr a célula anterior (extração do ZIP).")

In [ ]:
import yaml

yaml_candidates = sorted(Path("/content/data").rglob("data.yaml"))
if not yaml_candidates:
    raise FileNotFoundError(
        "data.yaml não encontrado em /content/data. Rode a célula do ZIP antes e confira se apareceu "
        "'data.yaml encontrados: [...]' com pelo menos um caminho."
    )


def parece_dataset_yolo(p):
    base = p.parent
    return (base / "train" / "images").is_dir() or (base / "train" / "images").exists()


bons = [p for p in yaml_candidates if parece_dataset_yolo(p)]
DATA_YAML = bons[0] if bons else yaml_candidates[0]
if len(yaml_candidates) > 1:
    print("Vários data.yaml:", yaml_candidates)
    print("Escolhido:", DATA_YAML)
BASE = DATA_YAML.parent.resolve()

with open(DATA_YAML, "r") as f:
    cfg = yaml.safe_load(f)

def resolve_split(key, fallback):
    if key not in cfg or cfg[key] is None:
        return
    raw = cfg[key]
    p = Path(raw)
    if not p.is_absolute():
        p = (BASE / raw).resolve()
    if p.exists():
        cfg[key] = str(p)
    else:
        alt = (BASE / fallback).resolve()
        if alt.exists():
            cfg[key] = str(alt)

resolve_split("train", "train/images")
resolve_split("val", "valid/images")
resolve_split("test", "test/images")

FIXED_YAML = BASE / "data_colab.yaml"
with open(FIXED_YAML, "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Usando:", FIXED_YAML)
print(yaml.safe_dump(cfg, sort_keys=False))

## 4. Treino — Modelo 1 (detecção de mudas)

O relatório cita **RF-DETR Small**. No Colab, o caminho mais estável para o dataset **já em formato YOLO** é **YOLO11** (`ultralytics`). Isso cobre mAP, precision, recall no validador.

Depois do treino, os pesos ficam em `.../weights/best.pt` (pasta do `run` do Ultralytics). **Obrigatório no Colab:** *Ambiente → Alterar tipo de ambiente* e escolher **GPU**; sem isso o log mostra `device=cpu` e o treino demora horas.

**Critério "germinou" (por imagem de bandeja)**: presença de pelo menos uma detecção das classes de interesse (ex.: `seedling`, `twoseedling`, conforme seu `data.yaml`). Ajuste `GERMINATION_CLASS_IDS` abaixo.

In [ ]:
from ultralytics import YOLO
import os

# Evita pasta duplicada runs/detect/runs/detect se o cwd não for /content
os.chdir("/content")

if not torch.cuda.is_available():
    print("=" * 60)
    print("AVISO: estás a treinar em CPU — será MUITO lento.")
    print("No Colab: Ambiente → Alterar tipo de ambiente → acelerador GPU (T4/L4/A100)")
    print("Depois: Ambiente → Reiniciar sessão e voltar a correr as células desde o início.")
    print("=" * 60)

# Ajuste conforme seu data.yaml (índices 0..nc-1)
GERMINATION_CLASS_IDS = None  # None = todas as classes contam como "planta"

_dev = 0 if torch.cuda.is_available() else "cpu"
_workers = 4 if torch.cuda.is_available() else 2

model = YOLO("yolo11s.pt")
results = model.train(
    data=str(FIXED_YAML),
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    device=_dev,
    workers=_workers,
    project="/content/runs/detect",
    name="seedling_yolo11",
    exist_ok=True,
)

best_pt = Path(results.save_dir) / "weights" / "best.pt"
print("Melhor peso:", best_pt)

### Validação explícita (mAP, precision, recall)

In [ ]:
metrics = model.val(data=str(FIXED_YAML), split="val")
print(metrics)

### (Opcional) RF-DETR Small — treino fora do Ultralytics

Para reproduzir literalmente o modelo do relatório, instale o projeto open-source da Roboflow e siga o README de treino com dataset **COCO**. Fluxo típico: converter labels YOLO → COCO → treinar RF-DETR. Isso é mais trabalhoso que YOLO11; reserve uma sessão só para essa conversão.

```
!pip install -q rfdetr
# ou: clone https://github.com/roboflow/rf-detr e siga os scripts de treino
```

## 5. Inferência + recortes para o Modelo 2 (folhas)

Para cada bounding box de muda, gera um recorte RGB usado depois pelo **contador de folhas** (regressão).

In [ ]:
import cv2
import numpy as np
from PIL import Image

detector = YOLO(str(best_pt))

# Use uma imagem do seu valid ou test
with open(FIXED_YAML, "r") as f:
    cfg = yaml.safe_load(f)
val_dir = Path(cfg["val"]).parent
sample_imgs = sorted(val_dir.glob("*.jpg")) + sorted(val_dir.glob("*.png"))
assert sample_imgs, "Nenhuma imagem no conjunto de validação."
img_path = sample_imgs[0]

res = detector.predict(source=str(img_path), conf=0.25, verbose=False)[0]
im_bgr = cv2.imread(str(img_path))
h, w = im_bgr.shape[:2]

crops_dir = Path("/content/crops")
crops_dir.mkdir(exist_ok=True)

for i, box in enumerate(res.boxes):
    cls_id = int(box.cls[0])
    if GERMINATION_CLASS_IDS is not None and cls_id not in GERMINATION_CLASS_IDS:
        continue
    xyxy = box.xyxy[0].cpu().numpy().astype(int)
    x1, y1, x2, y2 = [max(0, v) for v in xyxy]
    crop = im_bgr[y1:y2, x1:x2]
    out_p = crops_dir / f"muda_{i}.jpg"
    cv2.imwrite(str(out_p), crop)

print("Recortes salvos em", crops_dir, "— quantidade:", len(list(crops_dir.glob('*.jpg'))))

## 6. Modelo 2 — contagem de folhas (esqueleto PyTorch)

Você precisa de um CSV com colunas `[caminho_imagem, num_folhas]` (anotação manual em subconjunto de recortes). Sem rótulos reais, não há como treinar com significado — o bloco abaixo mostra a **arquitetura sugerida no relatório**: backbone ImageNet + cabeça de regressão.

Substitua `CSV_LABELS` pelo seu arquivo quando tiver anotações.

In [ ]:
import pandas as pd
import torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader

CSV_LABELS = Path("/content/leaf_counts.csv")  # crie este arquivo

class LeafCountCSV(Dataset):
    def __init__(self, csv_path, img_root=None):
        self.df = pd.read_csv(csv_path)
        self.img_root = Path(img_root) if img_root else None
        self.tfms = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = Path(row["path"])
        if self.img_root:
            p = self.img_root / p
        y = float(row["num_folhas"])
        im = Image.open(p).convert("RGB")
        return self.tfms(im), torch.tensor(y, dtype=torch.float32)


class LeafRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        w = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        self.backbone = w.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(576, 128),
            nn.ReLU(inplace=True),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x)
        return self.head(x).squeeze(-1)


if CSV_LABELS.exists():
    ds = LeafCountCSV(CSV_LABELS)
    dl = DataLoader(ds, batch_size=8, shuffle=True, num_workers=2)
    net = LeafRegressor().cuda() if torch.cuda.is_available() else LeafRegressor()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    loss_fn = nn.SmoothL1Loss()
    for epoch in range(3):
        running = 0.0
        for xb, yb in dl:
            if torch.cuda.is_available():
                xb, yb = xb.cuda(), yb.cuda()
            pred = net(xb)
            loss = loss_fn(pred, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            running += loss.item() * xb.size(0)
        print(f"epoch {epoch} loss {running/len(ds):.4f}")
else:
    print("Crie", CSV_LABELS, "com colunas path,num_folhas para treinar o contador.")

## 7. Métricas de contagem (MAE / MSE) — exemplo

Quando tiver predições `y_pred` e rótulos `y_true`:

```python
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae = mean_absolute_error(y_true, y_pred)
mse = mean_squared_error(y_true, y_pred)
```

## 8. Série temporal (germinação ao longo dos dias)

Organize pastas `D0`, `D1`, … com fotos da mesma bandeja. Para cada dia, rode o detector, conte detecções por classe e exporte um CSV para gráficos.

Exemplo de agregação:

In [ ]:
def summarize_image(path, detector, germination_classes=None):
    r = detector.predict(source=str(path), verbose=False)[0]
    n = 0
    for box in r.boxes:
        cid = int(box.cls[0])
        if germination_classes is None or cid in germination_classes:
            n += 1
    return {"path": str(path), "num_plantas_detectadas": n, "germinou": n > 0}

# Exemplo com uma única imagem
pd.DataFrame([summarize_image(img_path, detector)])

## 9. Visualização com Supervision (opcional)

Útil para qualidade das detecções em artigos/apresentações.

In [ ]:
import supervision as sv

res = detector.predict(source=str(img_path), verbose=False)[0]
detections = sv.Detections.from_ultralytics(res)
annotated = sv.BoxAnnotator().annotate(cv2.imread(str(img_path)), detections)
sv.plot_image(annotated, size=(12, 12))

---

### Checklist alinhado ao relatório

- [ ] Dataset YOLO exportado do Roboflow + `data.yaml` corrigido neste notebook.
- [ ] Treino Modelo 1 + métricas mAP/precision/recall no valid.
- [ ] CSV de folhas por recorte para treinar Modelo 2.
- [ ] Avaliação MAE/MSE da contagem vs anotação manual.
- [ ] (Opcional) Pastas por dia para curvas de germinação/crescimento.

**Referências no documento**: Roboflow seedlings, leaf counting (LC-Net / regressão), pipeline temporal e independência de créditos de treino usando Colab.